In [2]:
from IPython.display import display
import ast
import numpy as np
import pandas as pd
from scipy import stats

import main_analysis as ma

RESULTS_DIR = 'results'

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


In [5]:
core_frois = ['FFA', 'PPA', 'EBA']
all_frois = core_frois + ['OFA', 'FBA', 'OPA', 'VWFA', 'OVWFA']

to_model_names = ma.get_model_names('task_optimized')
random_model_names = ma.get_model_names('untrained')
print(f'Found {len(to_model_names)} task-optimized models and {len(random_model_names)} random models.')
standard_model_name = 'resnet50imagenet1kv2'
best_model_name = 'vitb16siglip2res224'
examplerandom_model_name = 'alexnetrandom5'

Found 35 task-optimized models and 20 random models.


# Section 2.1

In [ ]:
LOCALIZATION_VALIDATION_CSV_PATH = f'{RESULTS_DIR}/localization_validation_grillspector_val_prince2024.csv'

tthreshold = 7
random_tthreshold = 3

model_localizer = 'grillspector'

Found 35 task-optimized models and 20 random models.


In [4]:
locval_df = pd.read_csv(LOCALIZATION_VALIDATION_CSV_PATH)

standard_model_rows = locval_df[
    (locval_df['model_name'] == standard_model_name) &
    (locval_df['model_localization_localizer'] == model_localizer) &
    (locval_df['model_froi_tthreshold'] == tthreshold) &
    (locval_df['model_froi'].isin(core_frois))
]
standard_model_u_pvals = []
for _, srow in standard_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(srow['validation_u_pvalues'])
    standard_model_u_pvals += row_u_pvals
print(f'Standard model {standard_model_name} has maximum p-value {np.max(standard_model_u_pvals)}')

best_model_rows = locval_df[
    (locval_df['model_name'] == best_model_name) &
    (locval_df['model_localization_localizer'] == model_localizer) &
    (locval_df['model_froi_tthreshold'] == tthreshold) &
    (locval_df['model_froi'].isin(core_frois))
]
best_model_u_pvals = []
for _, brow in best_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(brow['validation_u_pvalues'])
    best_model_u_pvals += row_u_pvals
print(f'Best model {best_model_name} has maximum p-value {np.max(best_model_u_pvals)}')

to_model_rows = locval_df[
    (locval_df['model_name'].isin(to_model_names)) &
    (locval_df['model_localization_localizer'] == model_localizer) &
    (locval_df['model_froi_tthreshold'] == tthreshold) &
    (locval_df['model_froi'].isin(core_frois))
]
to_model_u_pvals = []
for _, trow in to_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(trow['validation_u_pvalues'])
    to_model_u_pvals += row_u_pvals
print(f'Task-optimized models have maximum p-value {np.max(to_model_u_pvals)}')

print('\n\n')
examplerandom_model_rows = locval_df[
    (locval_df['model_name'] == examplerandom_model_name) &
    (locval_df['model_localization_localizer'] == model_localizer) &
    (locval_df['model_froi_tthreshold'] == random_tthreshold) &
    (locval_df['model_froi'].isin(core_frois))
]
examplerandom_model_u_pvals = []
for _, errow in examplerandom_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(errow['validation_u_pvalues'])
    print(f'froi {errow["model_froi"]} p-values: {row_u_pvals}')

print('\n\n')
untrained_model_rows = locval_df[
    (locval_df['model_name'].isin(random_model_names)) &
    (locval_df['model_localization_localizer'] == model_localizer) &
    (locval_df['model_froi_tthreshold'] == random_tthreshold) &
    (locval_df['model_froi'].isin(core_frois))
]
untrained_model_u_pvals = []
n_untrained_frois = {froi: 0 for froi in core_frois}
for _, urrow in untrained_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(urrow['validation_u_pvalues'])
    all_pval_lt_0pt05 = all([pval < 0.05 for pval in row_u_pvals])
    if all_pval_lt_0pt05:
        n_untrained_frois[urrow['model_froi']] += 1
        print(f'Untrained model {urrow["model_name"]} has p<0.05 in all runs for fROI {urrow["model_froi"]}: {row_u_pvals}')
    untrained_model_u_pvals += row_u_pvals
    # print(row_u_pvals)

print('\n\n')
for _, urrow in untrained_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(urrow['validation_u_pvalues'])
    if urrow["model_name"] == examplerandom_model_name:
        print(f'Example random model {urrow["model_name"]} fROI {urrow["model_froi"]} p-values: {row_u_pvals}')

print('\n\n')
print(f'Untrained models have maximum p-value {np.max(untrained_model_u_pvals)}')
print(f'Number of untrained models with p<0.05 in all fROIs: {n_untrained_frois}')

Standard model resnet50imagenet1kv2 has maximum p-value 6.875277155242618e-19
Best model vitb16siglip2res224 has maximum p-value 1.0046503105672565e-22
Task-optimized models have maximum p-value 0.028550648235675383



froi EBA p-values: [1.8566716796386583e-07, 6.518249224950052e-21, 3.106691374822264e-15]
froi FFA p-values: [0.7979680061661435, 4.663211238633869e-10, 2.139183884285796e-08]
froi PPA p-values: [0.44099425935491926, 4.548307034099951e-12, 0.6694958320262913]



Untrained model alexnetrandom1 has p<0.05 in all runs for fROI PPA: [0.0072790559612759195, 1.4857634787857233e-11, 8.432285519074714e-05]
Untrained model alexnetrandom1 has p<0.05 in all runs for fROI EBA: [0.04741255354799484, 2.87031675133382e-24, 2.8296226491843953e-16]
Untrained model alexnetrandom2 has p<0.05 in all runs for fROI EBA: [3.244822427529779e-06, 3.1025332840738693e-20, 5.975445802834818e-14]
Untrained model alexnetrandom2 has p<0.05 in all runs for fROI PPA: [1.1528265957795565e-14, 2.221008976

# Section 2.2

In [ ]:
SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/modelsubject_correlations_nsd515.csv'
FROIFROI_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/froifroi_correlations_nsd515.csv'

tthreshold = 7

model_localizer = 'grillspector'

metrics = ['normed_pearsonr', 'normed_rdmr']

## Human-Human

In [6]:
hh_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
hh_csv = hh_csv[
    (hh_csv['fmri_froi'].isin(core_frois)) &
    (hh_csv['fmri_tthreshold'] == tthreshold)
]
print(f'Minimum normed pearsonr across core ROIs and across all subject pairs:')
print(hh_csv['normed_pearsonr'].min().item())
print(f'Maximum normed pearsonr p-value across core ROIs and across all subject pairs:')
print(hh_csv['pvalue_pearsonr'].max().item())
print(f'Minimum normed rdmr across core ROIs and across all subject pairs:')
print(hh_csv['normed_rdmr'].min().item())
print(f'Maximum normed rdmr p-value across core ROIs and across all subject pairs:')
print(hh_csv['pvalue_rdmr'].max().item())
# hh_csv_grouped = hh_csv.groupby(['fmri_froi']).aggregate(
#     {
#         'normed_pearsonr': ['median'],
#         'normed_rdmr': ['median'],
#     }
# )
hh_csv_grouped = (
    hh_csv
    .groupby(['fmri_froi'])
    .agg(
        normed_pearsonr_median=('normed_pearsonr', 'median'),
        normed_rdmr_median=('normed_rdmr', 'median'),
    )
    .reset_index()
).set_index('fmri_froi')
display(hh_csv_grouped)
print('Median normed pearsonr across core ROIs:')
print(hh_csv_grouped['normed_pearsonr_median'].median().item())
print('Median normed rdmr across core ROIs:')
print(hh_csv_grouped['normed_rdmr_median'].median().item())

Minimum normed pearsonr across core ROIs and across all subject pairs:
0.8727959208707898
Maximum normed pearsonr p-value across core ROIs and across all subject pairs:
1.4775544305207639e-38
Minimum normed rdmr across core ROIs and across all subject pairs:
0.8050828196815112
Maximum normed rdmr p-value across core ROIs and across all subject pairs:
0.0


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.997637,1.127444
FFA,0.983508,1.009670
PPA,1.014474,1.069014


Median normed pearsonr across core ROIs:
0.997636511462672
Median normed rdmr across core ROIs:
1.0690142237039242


### Human-Human: Median of medians

In [10]:
hh_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
subjects = list(range(1, 9)) # Subjects are 1-indexed
subject_othersubjects_medians = {}
metrics = ['normed_pearsonr', 'normed_rdmr']
frois = core_frois

for metric in metrics:
    subject_othersubjects_medians[metric] = {}
    for froi in frois:
        subject_othersubjects_medians[metric][froi] = []
        for subject in subjects:
            hh_medians = []
            subjectsubject_froi = hh_csv[
                (hh_csv['fmri_froi'] == froi) &
                (hh_csv['fmri_tthreshold'] == tthreshold) &
                (
                    (hh_csv['subject1'] == subject) |
                    (hh_csv['subject2'] == subject)
                )
            ]
            assert len(subjectsubject_froi) == 7, f'Expected 7 rows for subject {subject} and froi {froi}, but got {len(subjectsubject_froi)}'
            subject_othersubjects_medians[metric][froi].append(subjectsubject_froi[metric].median().item()) # Append the median for this subject

for metric in metrics:
    print(f'Median {metric} across subjects for each froi:')
    for froi in frois:
        froi_medians = subject_othersubjects_medians[metric][froi]
        print(f'  {froi}: median={np.median(froi_medians):.4f}, sem={stats.sem(froi_medians):.4f}')

print('\n')
# Print median across frois for each metric
for metric in metrics:
    all_froi_medians = []
    for froi in frois:
        all_froi_medians += subject_othersubjects_medians[metric][froi]
    print(f'Median {metric} across subjects and frois: median={np.median(all_froi_medians):.4f}, sem={stats.sem(all_froi_medians):.4f}')

Median normed_pearsonr across subjects for each froi:
  FFA: median=0.9812, sem=0.0092
  PPA: median=1.0059, sem=0.0085
  EBA: median=0.9919, sem=0.0128
Median normed_rdmr across subjects for each froi:
  FFA: median=0.9965, sem=0.0210
  PPA: median=1.0222, sem=0.0282
  EBA: median=1.0910, sem=0.0286


Median normed_pearsonr across subjects and frois: median=0.9927, sem=0.0065
Median normed_rdmr across subjects and frois: median=1.0262, sem=0.0167


## Model-Human

In [18]:
mh_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)
mh_csv = mh_csv[
    (mh_csv['fmri_froi'].isin(core_frois)) &
    (mh_csv['fmri_tthreshold'] == tthreshold) &
    (mh_csv['model_name'].isin(to_model_names)) &
    (mh_csv['model_localizer'] == model_localizer) &
    (mh_csv['model_froi'].isin(core_frois)) &
    (mh_csv['model_tthreshold'] == tthreshold)
]



mh_csv_grouped = (
    mh_csv
    .groupby(['fmri_froi', 'model_name'])
    .agg(
        normed_pearsonr_median=('normed_pearsonr', 'median'),
        normed_rdmr_median=('normed_rdmr', 'median'),
    )
    .reset_index()
)

mh_csv_grouped_best = mh_csv_grouped[
    mh_csv_grouped['model_name'] == best_model_name
]
# drop the model_name column & set fmri_froi as index
mh_csv_grouped_best = mh_csv_grouped_best.drop(columns=['model_name']).set_index('fmri_froi')
print(f'Model-Subject correlations for best model: {best_model_name}')
display(mh_csv_grouped_best)

mh_csv_grouped2 = (
    mh_csv_grouped
    .groupby('fmri_froi')
    .agg(
        normed_pearsonr_median=('normed_pearsonr_median', 'median'),
        normed_rdmr_median=('normed_rdmr_median', 'median'),
    )
    .reset_index()
).set_index('fmri_froi')
print('Model-Subject correlations (grouped by ROI):')
display(mh_csv_grouped2)

print('Delta Rs between model-subject and subject-subject medians:')
# delta Rs between model-subject and subject-subject medians
display(hh_csv_grouped - mh_csv_grouped2)

print(f'One sided U-test between subject-subject and all model-subject correlations:')
# u-test between subject-subject and model-subject correlations, for each metric and fROI
for froi in core_frois:
    print(f'\nFROI: {froi}')
    for metric in metrics:
        hh_values = hh_csv[
            (hh_csv['fmri_froi'] == froi)
        ][metric].values
        mh_values = mh_csv[
            (mh_csv['fmri_froi'] == froi)
        ][metric].values
        u_stat, p_value = stats.mannwhitneyu(hh_values, mh_values, alternative='greater')
        print(f'Metric: {metric}, U-statistic: {u_stat}, p-value: {p_value},    median subject-subject: {np.median(hh_values)}, median model-subject: {np.median(mh_values)}')

Model-Subject correlations for best model: vitb16siglip2res224


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.764139,0.481560
FFA,0.657793,0.574184
PPA,0.711801,0.310059


Model-Subject correlations (grouped by ROI):


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.672800,0.306611
FFA,0.618296,0.383438
PPA,0.726980,0.367889


Delta Rs between model-subject and subject-subject medians:


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.324837,0.820833
FFA,0.365212,0.626231
PPA,0.287494,0.701125


One sided U-test between subject-subject and all model-subject correlations:

FROI: FFA
Metric: normed_pearsonr, U-statistic: 7840.0, p-value: 1.3502757183221409e-18,    median subject-subject: 0.9835081012617362, median model-subject: 0.6185011731061831
Metric: normed_rdmr, U-statistic: 7840.0, p-value: 1.3502757183221409e-18,    median subject-subject: 1.0096695737095467, median model-subject: 0.4006275595388278

FROI: PPA
Metric: normed_pearsonr, U-statistic: 7840.0, p-value: 1.3502757183221409e-18,    median subject-subject: 1.014473855009372, median model-subject: 0.7248889889428041
Metric: normed_rdmr, U-statistic: 7840.0, p-value: 1.3502757183221409e-18,    median subject-subject: 1.0690142237039242, median model-subject: 0.3565286696947123

FROI: EBA
Metric: normed_pearsonr, U-statistic: 7840.0, p-value: 1.3502757183221409e-18,    median subject-subject: 0.997636511462672, median model-subject: 0.6662167414467046
Metric: normed_rdmr, U-statistic: 7840.0, p-value: 1.350275718322

### ANN-brain < brain-brain corrs

In [8]:
pvalues_dict = {}
model_medians_dict = {}
for metric in metrics:
    metric_min_pval = np.inf
    pvalues_dict[metric] = {}
    model_medians_dict[metric] = {}
    print(f'For metric: {metric}')
    for froi in core_frois:
        pvalues_dict[metric][froi] = {}
        model_medians_dict[metric][froi] = {}
        print(f'  For ROI: {froi}')
        hh_subset = hh_csv[
            (hh_csv['fmri_froi'] == froi)
        ]
        hh_corrs = hh_subset[metric].values
        for model_name in to_model_names:
            subset = mh_csv[
                (mh_csv['fmri_froi'] == froi) &
                (mh_csv['model_name'] == model_name)
            ]
            mh_corrs = subset[metric].values
            u_stat, p_value = stats.mannwhitneyu(
                mh_corrs,
                hh_corrs,
                alternative='less'
            )
            if p_value >= 0.05:
                print(f'    Model {model_name} p-value: {p_value} (NOT SIGNIFICANT)')
            # else:
            #     print(f'    Model {model_name} p-value: {p_value} (SIGNIFICANT)')
            pvalues_dict[metric][froi][model_name] = p_value
            model_medians_dict[metric][froi][model_name] = np.median(mh_corrs)
            if p_value < metric_min_pval:
                metric_min_pval = p_value
        metric_froi_best_model_median = np.median(model_medians_dict[metric][froi][best_model_name])
        metric_froi_model_medians = model_medians_dict[metric][froi].values()
        median_of_medians = np.median(list(metric_froi_model_medians))
        print(f'    Median of model medians for ROI {froi} and metric {metric}: {median_of_medians}')
        # Print the median for the best model
        print(f'    Median of {best_model_name} for ROI {froi} and metric {metric}: {metric_froi_best_model_median}')
    print('\n')
    print(f'Minimum p-value for metric {metric} across core ROIs and models: {metric_min_pval}')
    print('\n')

For metric: normed_pearsonr
  For ROI: FFA
    Median of model medians for ROI FFA and metric normed_pearsonr: 0.6182962524942974
    Median of vitb16siglip2res224 for ROI FFA and metric normed_pearsonr: 0.6577928973056599
  For ROI: PPA
    Median of model medians for ROI PPA and metric normed_pearsonr: 0.7269798643138887
    Median of vitb16siglip2res224 for ROI PPA and metric normed_pearsonr: 0.7118005729011971
  For ROI: EBA
    Median of model medians for ROI EBA and metric normed_pearsonr: 0.6727995765455339
    Median of vitb16siglip2res224 for ROI EBA and metric normed_pearsonr: 0.7641392044521946


Minimum p-value for metric normed_pearsonr across core ROIs and models: 3.3046555326212464e-08


For metric: normed_rdmr
  For ROI: FFA
    Median of model medians for ROI FFA and metric normed_rdmr: 0.3834384870617388
    Median of vitb16siglip2res224 for ROI FFA and metric normed_rdmr: 0.5741843944220171
  For ROI: PPA
    Median of model medians for ROI PPA and metric normed_rdmr

In [30]:
# Subtract the grouped model-subject correlations from the grouped subject-subject correlations
print('GAP between subject-subject and model-subject correlations (grouped by ROI):')
median_gap_df = hh_csv_grouped - mh_csv_grouped2
display(median_gap_df)

print(f'Model-Subject correlations for best model: {best_model_name}')
display(mh_csv_grouped_best)

print(f'GAP between subject-subject and model-subject correlations for the best model {best_model_name} (grouped by ROI):')
best_gap_df = hh_csv_grouped - mh_csv_grouped_best
display(best_gap_df)

GAP between subject-subject and model-subject correlations (grouped by ROI):


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.324837,0.820833
FFA,0.365212,0.626231
PPA,0.287494,0.701125


Model-Subject correlations for best model: vitb16siglip2res224


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.764139,0.481560
FFA,0.657793,0.574184
PPA,0.711801,0.310059


GAP between subject-subject and model-subject correlations for the best model vitb16siglip2res224 (grouped by ROI):


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.233497,0.645884
FFA,0.325715,0.435485
PPA,0.302673,0.758955


## fROI-fROI

In [5]:
froifroi_df = pd.read_csv(FROIFROI_CORRELATIONS_CSV_PATH)

In [6]:
# For each fmri_froi1, fmri_froi2 pair, compute the median correlation (for both metrics)
froifroi_grouped = (
    froifroi_df
    .groupby(['fmri_froi1', 'fmri_froi2'])
    .agg(
        normed_pearsonr_median=('normed_pearsonr', 'median'),
        normed_rdmr_median=('normed_rdmr', 'median'),
    )
    .reset_index()
).set_index(['fmri_froi1', 'fmri_froi2'])
display(froifroi_grouped)

extra_frois = ['OFA', 'FBA', 'OPA']
all_frois = core_frois + extra_frois
for metric in metrics:
    print(f'Median {metric} between fROIs:')
    for cfroi in core_frois:
        cfroi_cfroi_corr = froifroi_grouped.loc[(cfroi, cfroi), f'{metric}_median']
        for efroi in all_frois:
            if cfroi == efroi:
                continue
            try:
                cfroi_efroi_corr = froifroi_grouped.loc[(cfroi, efroi), f'{metric}_median']
            except KeyError:
                cfroi_efroi_corr = froifroi_grouped.loc[(efroi, cfroi), f'{metric}_median']
            print(f'  {cfroi} - {efroi}: {cfroi_efroi_corr} (cfroi-cfroi: {cfroi_cfroi_corr}, delta: {cfroi_cfroi_corr - cfroi_efroi_corr})')
        print('\n')

# create a matrix for delta median correlations between fROIs (for both metrics)
# for metric in metrics:
#     print(f'Delta median {metric} between fROIs:')
#     delta_matrix = pd.DataFrame(index=selected_frois, columns=selected_frois)
#     for froi1 in selected_frois:
#         for froi2 in selected_frois:
#             if froi1 == froi2:
#                 delta_matrix.loc[froi1, froi2] = 0.0
#             else:
#                 try:
#                     median_corr = froifroi_grouped.loc[(froi1, froi2), f'{metric}_median']
#                 except KeyError:
#                     median_corr = froifroi_grouped.loc[(froi2, froi1), f'{metric}_median']
#                 delta_matrix.loc[froi1, froi2] = median_corr
#     display(delta_matrix)

normed_pearsonr_median  normed_rdmr_median
fmri_froi1 fmri_froi2                                            
EBA        EBA                       0.997636            1.127444
           FBA                       0.949230            1.009162
           OFA                       0.579433            0.544483
           OPA                      -0.507786            0.332061
           OVWFA                     0.122831            0.210786
           RSC                      -0.329870            0.136359
           VWFA                     -0.002721            0.383143
FBA        FBA                       0.947194            1.010690
           OVWFA                     0.218793            0.247331
           VWFA                      0.094470            0.399414
FFA        EBA                       0.784028            0.721792
           FBA                       0.805961            0.759663
           FFA                       0.983508            1.009670
           OFA                       0.902018            0.906203
           OPA                      -0.572707            0.366994
           OVWFA                     0.336205            0.249777
           PPA                      -0.539919            0.260137
           RSC                      -0.360563            0.118249
           VWFA                      0.084951            0.342663
OFA        FBA                       0.635573            0.586737
           OFA                       0.885005            0.867936
           OPA                      -0.445066            0.306830
           OVWFA                     0.526221            0.482688
           RSC                      -0.357431            0.126930
           VWFA                      0.290719            0.384783
OPA        FBA                      -0.521691            0.358696
           OPA                       1.004247            1.115832
           OVWFA                    -0.100597            0.188138
           RSC                       0.797298            0.711687
           VWFA                     -0.013293            0.259672
OVWFA      OVWFA                     0.724501            0.768379
PPA        EBA                      -0.520527            0.263552
           FBA                      -0.549094            0.311243
           OFA                      -0.440682            0.204225
           OPA                       0.977279            1.011848
           OVWFA                    -0.114596            0.110366
           PPA                       1.014474            1.069014
           RSC                       0.872965            0.827095
           VWFA                     -0.090037            0.178401
RSC        FBA                      -0.384803            0.167361
           OVWFA                    -0.193994            0.068670
           RSC                       0.987234            1.030239
           VWFA                     -0.252792            0.157013
VWFA       OVWFA                     0.566648            0.433717
           VWFA                      0.759721            0.849858

Median normed_pearsonr between fROIs:
  FFA - PPA: -0.5399186851710085 (cfroi-cfroi: 0.9835080579859774, delta: 1.523426743156986)
  FFA - EBA: 0.7840284402808035 (cfroi-cfroi: 0.9835080579859774, delta: 0.19947961770517386)
  FFA - OFA: 0.9020183466409057 (cfroi-cfroi: 0.9835080579859774, delta: 0.08148971134507166)
  FFA - FBA: 0.8059606598465057 (cfroi-cfroi: 0.9835080579859774, delta: 0.17754739813947173)
  FFA - OPA: -0.5727069731915267 (cfroi-cfroi: 0.9835080579859774, delta: 1.556215031177504)


  PPA - FFA: -0.5399186851710085 (cfroi-cfroi: 1.0144737554533512, delta: 1.5543924406243597)
  PPA - EBA: -0.5205267253040443 (cfroi-cfroi: 1.0144737554533512, delta: 1.5350004807573954)
  PPA - OFA: -0.4406817794284794 (cfroi-cfroi: 1.0144737554533512, delta: 1.4551555348818306)
  PPA - FBA: -0.549094208202048 (cfroi-cfroi: 1.0144737554533512, delta: 1.5635679636553992)
  PPA - OPA: 0.9772789675957122 (cfroi-cfroi: 1.0144737554533512, delta: 0.03719478785763897)


  EBA - FFA: 0.784028

# Section 2.4

In [6]:
tthresholds = [3, 5, 7, 9, 11]
model_localizer = 'grillspector'

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/tthreshold_effect_correlations_nsd515.csv'

## t-thresholds

In [31]:
hh_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
mh_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)

deltas_df = pd.DataFrame(columns=['fmri_froi', 'model_tthreshold', 'fmri_tthreshold', 'delta_normed_pearsonr_median', 'delta_normed_rdmr_median'])

for fmri_tthreshold in tthresholds:
    for model_tthreshold in tthresholds:
        hh_csv_subset = hh_csv[
            (hh_csv['fmri_froi'].isin(core_frois)) &
            (hh_csv['fmri_tthreshold'] == fmri_tthreshold)
        ]

        hh_csv_grouped = (
            hh_csv_subset
            .groupby(['fmri_froi'])
            .agg(
                normed_pearsonr_median=('normed_pearsonr', 'median'),
                normed_rdmr_median=('normed_rdmr', 'median'),
            )
            .reset_index()
        ).set_index('fmri_froi')
        # display(hh_csv_grouped)

        mh_csv_subset = mh_csv[
            (mh_csv['fmri_froi'].isin(core_frois)) &
            (mh_csv['fmri_tthreshold'] == fmri_tthreshold) &
            (mh_csv['model_name'].isin(to_model_names)) &
            (mh_csv['model_localizer'] == model_localizer) &
            (mh_csv['model_froi'].isin(core_frois)) &
            (mh_csv['model_tthreshold'] == model_tthreshold)
        ]
        mh_csv_grouped = (
            mh_csv_subset
            .groupby(['fmri_froi', 'model_name'])
            .agg(
                normed_pearsonr_median=('normed_pearsonr', 'median'),
                normed_rdmr_median=('normed_rdmr', 'median'),
            )
            .reset_index()
        )

        mh_csv_grouped_best = mh_csv_grouped[
            mh_csv_grouped['model_name'] == best_model_name
        ]
        # drop the model_name column & set fmri_froi as index
        mh_csv_grouped_best = mh_csv_grouped_best.drop(columns=['model_name']).set_index('fmri_froi')
        # display(mh_csv_grouped_best)

        # Subtract the grouped model-subject correlations from the grouped subject-subject correlations
        tthresholds_delta_df = hh_csv_grouped - mh_csv_grouped_best
        # Rename columns to indicate these are deltas
        tthresholds_delta_df = tthresholds_delta_df.rename(columns={
            'normed_pearsonr_median': 'delta_normed_pearsonr_median',
            'normed_rdmr_median': 'delta_normed_rdmr_median',
        })
        # display(tthresholds_delta_df)

        # Add columns for fmri_tthreshold and model_tthreshold
        tthresholds_delta_df = tthresholds_delta_df.reset_index()
        tthresholds_delta_df['fmri_tthreshold'] = fmri_tthreshold
        tthresholds_delta_df['model_tthreshold'] = model_tthreshold
        # Join with the deltas_df
        deltas_df = pd.concat([deltas_df, tthresholds_delta_df], ignore_index=True)

# Print lowest deltas for each metric (and the corresponding tthresholds) and for each froi
for metric in ['delta_normed_pearsonr_median', 'delta_normed_rdmr_median']:
    print(f'Lowest {metric} across all tthresholds and frois:')
    for froi in core_frois:
        froi_deltas = deltas_df[deltas_df['fmri_froi'] == froi]
        min_delta_row = froi_deltas.loc[froi_deltas[metric].idxmin()]
        print(f'  FROI: {froi}, Minimum {metric}: {min_delta_row[metric]:.4f} (fmri_tthreshold: {min_delta_row["fmri_tthreshold"]}, model_tthreshold: {min_delta_row["model_tthreshold"]})')
    print('\n')

Lowest delta_normed_pearsonr_median across all tthresholds and frois:
  FROI: FFA, Minimum delta_normed_pearsonr_median: 0.2832 (fmri_tthreshold: 11, model_tthreshold: 3)
  FROI: PPA, Minimum delta_normed_pearsonr_median: 0.2510 (fmri_tthreshold: 11, model_tthreshold: 3)
  FROI: EBA, Minimum delta_normed_pearsonr_median: 0.2263 (fmri_tthreshold: 11, model_tthreshold: 7)


Lowest delta_normed_rdmr_median across all tthresholds and frois:
  FROI: FFA, Minimum delta_normed_rdmr_median: 0.4135 (fmri_tthreshold: 11, model_tthreshold: 7)
  FROI: PPA, Minimum delta_normed_rdmr_median: 0.6990 (fmri_tthreshold: 11, model_tthreshold: 11)
  FROI: EBA, Minimum delta_normed_rdmr_median: 0.5583 (fmri_tthreshold: 11, model_tthreshold: 5)




/var/folders/f4/jrx4_ljx4_q3d25kvtsnw4tw0000gq/T/ipykernel_19352/2207149595.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  deltas_df = pd.concat([deltas_df, tthresholds_delta_df], ignore_index=True)


## Localizers

### Intersections

### Voxel-tuning matched

In [45]:
SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/modelsubject_correlations_nsd515.csv'

tthreshold = 7
model_localizer = 'nsd'

metrics = ['normed_pearsonr', 'normed_rdmr']

In [50]:
hh_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
mh_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)

deltas_df = pd.DataFrame(columns=['fmri_froi', 'model_tthreshold', 'fmri_tthreshold', 'delta_normed_pearsonr_median', 'delta_normed_rdmr_median'])

hh_csv_subset = hh_csv[
    (hh_csv['fmri_froi'].isin(core_frois)) &
    (hh_csv['fmri_tthreshold'] == tthreshold)
]

hh_csv_grouped = (
    hh_csv_subset
    .groupby(['fmri_froi'])
    .agg(
        normed_pearsonr_median=('normed_pearsonr', 'median'),
        normed_rdmr_median=('normed_rdmr', 'median'),
    )
    .reset_index()
).set_index('fmri_froi')
# display(hh_csv_grouped)

mh_csv_subset = mh_csv[
    (mh_csv['fmri_froi'].isin(core_frois)) &
    (mh_csv['fmri_tthreshold'] == tthreshold) &
    (mh_csv['model_name'].isin(to_model_names)) &
    (mh_csv['model_localizer'] == model_localizer) &
    (mh_csv['model_froi'].isin(core_frois)) &
    (mh_csv['model_tthreshold'] == tthreshold)
]
mh_csv_grouped = (
    mh_csv_subset
    .groupby(['fmri_froi', 'model_name'])
    .agg(
        normed_pearsonr_median=('normed_pearsonr', 'median'),
        normed_rdmr_median=('normed_rdmr', 'median'),
    )
    .reset_index()
)

mh_csv_grouped_best = mh_csv_grouped[
    mh_csv_grouped['model_name'] == best_model_name
]
# drop the model_name column & set fmri_froi as index
mh_csv_grouped_best = mh_csv_grouped_best.drop(columns=['model_name']).set_index('fmri_froi')
# display(mh_csv_grouped_best)

print(f'Delta Rs for best model: {best_model_name}')
# Subtract the grouped model-subject correlations from the grouped subject-subject correlations
best_delta_df = hh_csv_grouped - mh_csv_grouped_best
display(best_delta_df)

mh_csv_grouped2 = (
    mh_csv_grouped
    .groupby('fmri_froi')
    .agg(
        normed_pearsonr_median=('normed_pearsonr_median', 'median'),
        normed_rdmr_median=('normed_rdmr_median', 'median'),
    )
    .reset_index()
).set_index('fmri_froi')
median_delta_df = hh_csv_grouped - mh_csv_grouped2
print('Delta Rs between model-subject and subject-subject medians:')
display(median_delta_df)


Delta Rs for best model: vitb16siglip2res224


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.274721,0.618746
FFA,0.199050,0.380796
PPA,0.324131,0.766457


Delta Rs between model-subject and subject-subject medians:


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.259254,0.734399
FFA,0.225282,0.491223
PPA,0.223101,0.621829


# Section 2.5

In [ ]:
em_csv = pd.read_csv(f'{RESULTS_DIR}/layer_froi_lesion_ems_fixedalpha.csv')
em_csv2_tthresholds = pd.read_csv(f'{RESULTS_DIR}/nsd515_lesion_ems_fixedalpha.csv')

### Mean performance drop

In [38]:
mapper = 'ridge'
alpha = 0.1

em_csv_subset = em_csv[
    (em_csv['mapper'] == mapper) &
    (em_csv['alpha'] == alpha) &
    (em_csv['model_froi'].isin(core_frois)) &
    (em_csv['model_localizer'] == model_localizer) &
    (em_csv['model_tthreshold'] == tthreshold)
]

# Estimate mean and median of difference between layer_raw_pearsonr and lesion_raw_pearsonr, as well as for rdmr
em_csv_subset['delta_pearsonr'] = (em_csv_subset['layer_raw_pearsonr'] - em_csv_subset['lesion_raw_pearsonr'])/em_csv_subset['fmri_splithalf_pearsonr']*100
em_csv_subset['delta_rdmr'] = (em_csv_subset['layer_raw_rdmr'] - em_csv_subset['lesion_raw_rdmr'])/em_csv_subset['fmri_splithalf_rdmr']*100
print(f'Median delta pearsonr: {em_csv_subset["delta_pearsonr"].median()}')
print(f'Mean delta pearsonr: {em_csv_subset["delta_pearsonr"].mean()}')
print(f'Median delta rdmr: {em_csv_subset["delta_rdmr"].median()}')
print(f'Mean delta rdmr: {em_csv_subset["delta_rdmr"].mean()}')

print('\n')

# Estimate the same between layer_raw_pearsonr and froi_raw_pearsonr, as well as for rdmr
em_csv_subset['delta_pearsonr_froi'] = (em_csv_subset['layer_raw_pearsonr'] - em_csv_subset['froi_raw_pearsonr'])/em_csv_subset['fmri_splithalf_pearsonr']*100
em_csv_subset['delta_rdmr_froi'] = (em_csv_subset['layer_raw_rdmr'] - em_csv_subset['froi_raw_rdmr'])/em_csv_subset['fmri_splithalf_rdmr']*100
print(f'Median delta pearsonr (layer vs froi): {em_csv_subset["delta_pearsonr_froi"].median()}')
print(f'Mean delta pearsonr (layer vs froi): {em_csv_subset["delta_pearsonr_froi"].mean()}')
print(f'Median delta rdmr (layer vs froi): {em_csv_subset["delta_rdmr_froi"].median()}')
print(f'Mean delta rdmr (layer vs froi): {em_csv_subset["delta_rdmr_froi"].mean()}')

Median delta pearsonr: 0.19191641595982006
Mean delta pearsonr: 0.43291650939645926
Median delta rdmr: 0.4964057451717093
Mean delta rdmr: 1.194150102388364


Median delta pearsonr (layer vs froi): 18.628773099008043
Mean delta pearsonr (layer vs froi): 22.936058903588375
Median delta rdmr (layer vs froi): 32.69058724226957
Mean delta rdmr (layer vs froi): 34.021186787241334


/var/folders/f4/jrx4_ljx4_q3d25kvtsnw4tw0000gq/T/ipykernel_66432/1475731778.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  em_csv_subset['delta_pearsonr'] = (em_csv_subset['layer_raw_pearsonr'] - em_csv_subset['lesion_raw_pearsonr'])/em_csv_subset['fmri_splithalf_pearsonr']*100
/var/folders/f4/jrx4_ljx4_q3d25kvtsnw4tw0000gq/T/ipykernel_66432/1475731778.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  em_csv_subset['delta_rdmr'] = (em_csv_subset['layer_raw_rdmr'] - em_csv_subset['lesion_raw_rdmr'])

### Stats

In [ ]:
test_normality = False

mapper = 'ridge'
alpha = 0.1

em_csv_subset = em_csv[
    (em_csv['mapper'] == mapper) &
    (em_csv['alpha'] == alpha) &
    (em_csv['model_froi'].isin(core_frois)) &
    (em_csv['model_localizer'] == model_localizer) &
    (em_csv['model_tthreshold'] == tthreshold)
]

em_metrics = ['raw_pearsonr', 'raw_rdmr']

for metric in em_metrics:
    print(f'Metric: {metric}')
    for froi in core_frois:
        froi_subset = em_csv_subset[
            (em_csv_subset['model_froi'] == froi) &
            (em_csv_subset['fmri_froi'] == froi)
        ]
        # Group by model and take median across subjects for each metric
        froi_subset_grouped = (
            froi_subset
            .groupby(['model'])
            .agg(
                **{
                    f'layer_{metric}': (f'layer_{metric}', 'median'),
                    f'lesion_{metric}': (f'lesion_{metric}', 'median'),
                    f'froi_{metric}': (f'froi_{metric}', 'median'),
                }
            )
            .reset_index()
        )
        assert len(froi_subset_grouped) == len(to_model_names), f'Expected {len(to_model_names)} models in the grouped subset for froi {froi}, but got {len(froi_subset_grouped)}'
        layer_metric = froi_subset_grouped[f'layer_{metric}']
        lesion_metric = froi_subset_grouped[f'lesion_{metric}']
        froi_metric = froi_subset_grouped[f'froi_{metric}']
        print(f'  FROI: {froi}')
        if test_normality:
            # Do a test of normality for each metric
            layer_normality_pval = stats.shapiro(layer_metric).pvalue
            lesion_normality_pval = stats.shapiro(lesion_metric).pvalue
            froi_normality_pval = stats.shapiro(froi_metric).pvalue
            # Print p-values and the interpretation
            print(f'    Layer {metric} normality p-value: {layer_normality_pval} -> {"normal" if layer_normality_pval >= 0.05 else "not normal"}')
            print(f'    Lesion {metric} normality p-value: {lesion_normality_pval} -> {"normal" if lesion_normality_pval >= 0.05 else "not normal"}')
            print(f'    FROI {metric} normality p-value: {froi_normality_pval} -> {"normal" if froi_normality_pval >= 0.05 else "not normal"}')
        # Do a paired Wilcoxon signed-rank test between layer and lesion metrics, and between layer and froi metrics
        wilcoxon_layer_lesion_stat, wilcoxon_layer_lesion_pval = stats.wilcoxon(layer_metric, lesion_metric, alternative='greater')
        wilcoxon_layer_froi_stat, wilcoxon_layer_froi_pval = stats.wilcoxon(layer_metric, froi_metric, alternative='greater')
        print(f'    Wilcoxon layer vs lesion {metric}: stat={wilcoxon_layer_lesion_stat}, p-value={wilcoxon_layer_lesion_pval}')
        print(f'    Wilcoxon layer vs froi {metric}: stat={wilcoxon_layer_froi_stat}, p-value={wilcoxon_layer_froi_pval}')
    print('\n')

Metric: raw_pearsonr
  FROI: FFA
    Wilcoxon layer vs lesion raw_pearsonr: stat=9854.0, p-value=7.135600655378373e-25
    Wilcoxon layer vs froi raw_pearsonr: stat=9744.0, p-value=7.386606824496166e-24
  FROI: PPA
    Wilcoxon layer vs lesion raw_pearsonr: stat=9299.0, p-value=5.557665743376074e-20
    Wilcoxon layer vs froi raw_pearsonr: stat=9606.0, p-value=1.2881810440656058e-22
  FROI: EBA
    Wilcoxon layer vs lesion raw_pearsonr: stat=8234.0, p-value=3.3912156126094973e-12
    Wilcoxon layer vs froi raw_pearsonr: stat=9593.0, p-value=1.6792161297103249e-22


Metric: raw_rdmr
  FROI: FFA
    Wilcoxon layer vs lesion raw_rdmr: stat=9864.0, p-value=5.754921264551868e-25
    Wilcoxon layer vs froi raw_rdmr: stat=9631.0, p-value=7.721292045609256e-23
  FROI: PPA
    Wilcoxon layer vs lesion raw_rdmr: stat=9398.0, p-value=8.209676073262376e-21
    Wilcoxon layer vs froi raw_rdmr: stat=9648.0, p-value=5.443405331580223e-23
  FROI: EBA
    Wilcoxon layer vs lesion raw_rdmr: stat=9146.0,